In [ ]:
import requests
from typing import List, Dict, Optional
import pandas as pd
from sqlalchemy import create_engine,MetaData, Table, Column, Integer, String, ForeignKey , Numeric, Date , TEXT, Boolean
from dotenv import load_dotenv
import os
import json

In [ ]:
BASE_URL = "http://localhost:8000"

In [ ]:
MAX_ID_FILE = "max_ids.json"


def load_max_ids():
    if not os.path.exists(MAX_ID_FILE):
        with open(MAX_ID_FILE, "w") as f:
            json.dump({}, f)
        return {}

    with open(MAX_ID_FILE, "r") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return {}


def save_max_ids(max_ids: Dict[str, int]):
    with open(MAX_ID_FILE, "w") as f:
        json.dump(max_ids, f, indent=4)


def fetch_all_pages(endpoint: str, page_size: int = 10000):
    all_data = []
    page = 1

    max_ids = load_max_ids()

    last_max_id = max_ids.get(endpoint)

    while True:
        params = {
            "page": page,
            "page_size": page_size,
        }

        if last_max_id is not None:
            params["last_max_id"] = last_max_id

        response = requests.get(
            f"{BASE_URL}{endpoint}",
            params=params,
            timeout=10
        )

        if response.status_code != 200:
            print(f"Error {response.status_code}: {response.text}")
            break

        result = response.json()
        items = result.get("data", [])

        if not items:
            break

        all_data.extend(items)
        print(f"Fetched page {page} → {len(items)} records")

        last_item = items[-1]

        if "id" in last_item:
            last_max_id = last_item["id"]
        elif "user_id" in last_item:
            last_max_id = last_item["user_id"]

        if len(items) < page_size:
            break

        page += 1

    if last_max_id is not None:
        max_ids[endpoint] = last_max_id
        save_max_ids(max_ids)

    df = pd.DataFrame(all_data)
    print(f"Total records fetched: {len(all_data)}")
    return df